# SMS/HPS Baseline Timbre Transfer

This notebook demonstrates timbre transfer using the SMS (Spectral Modelling Synthesis) / HPS (Harmonic Plus Stochastic) vocoder baseline.

The SMS/HPS method decomposes audio into **harmonic** and **stochastic** components, then recombines them with swapped spectral envelopes to transfer timbre.

Requirements: `sms_tools`, `librosa`, `soundfile`, `scipy`

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent.parent
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")

from baseline import BaselineSMS
from utils import load_audio, save_audio, clip_audio, normalize_audio
from visualize import plot_spectrograms, plot_transfer_comparison

Project root: /m/home/home3/37/thieun1/unix/Project/final_project


/u/37/thieun1/unix/anaconda3/envs/conda_env3.10/lib/python3.10/site-packages/pyworld/__init__.py:13: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2026-04-04 14:06:31.867506: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-04 14:06:31.945342: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDN

In [2]:
# Edit these paths for your files
source_path = PROJECT_ROOT / 'data' / 'raw' / 'solo_violin' / 'VIII. Double.wav'
target_path = PROJECT_ROOT / "data/raw/voice/FULL/female1/scales/straight/f1_scales_straight_a.wav"

# Options: 'f0' or 'f0_ap'
method = 'f0_ap'
sample_rate = 16000
normalize_output = True
alpha = 1.0  # Blend strength: 0.0 = no transfer, 1.0 = full transfer

# SMS analysis parameters (defaults work well for most cases)
window = 'hann'
M = 1001       # Analysis window length (samples)
N = 2048       # FFT size
H = 256        # Hop size (samples)
n_harm = 20    # Max harmonics tracked per frame
min_f0 = 80.0  # Min F0 search range (Hz)
max_f0 = 1200.0  # Max F0 search range (Hz)
stoch_coeff = 128  # Stochastic residual coefficients

In [3]:
baseline_sms = BaselineSMS(
    sample_rate=sample_rate,
    window=window,
    M=M, N=N, H=H,
    n_harm=n_harm,
    min_f0=min_f0,
    max_f0=max_f0,
    stoch_coeff=stoch_coeff,
)

source_audio, _ = load_audio(source_path, sr=sample_rate, mono=True)
source_audio = clip_audio(source_audio, end_time=18.0, sr=sample_rate)
source_audio = normalize_audio(source_audio)
target_audio, _ = load_audio(target_path, sr=sample_rate, mono=True)
target_audio = normalize_audio(target_audio)

# SMS methods require source and target to have same length
target_audio, source_audio = baseline_sms.match_length(target_audio, source_audio)

source_audio = source_audio.astype('float64', copy=False)
target_audio = target_audio.astype('float64', copy=False)

print(f"Source: {len(source_audio)/sample_rate:.2f}s")
print(f"Target: {len(target_audio)/sample_rate:.2f}s")
print(f"Method: {method}, alpha: {alpha}")

if method == 'f0':
    output_audio = baseline_sms.f0_transfer(target_audio, source_audio, alpha=alpha)
elif method == 'f0_ap':
    output_audio = baseline_sms.f0_and_ap_transfer(target_audio, source_audio, alpha=alpha)
else:
    raise ValueError("method must be 'f0' or 'f0_ap'")

print(f"Output: {len(output_audio)/sample_rate:.2f}s")

ImportError: BaselineSMS requires the 'sms_tools' package. Install it or use the WORLD-based Baseline class instead.

In [ ]:
# Spectrograms: source, target, and transferred output
plot_transfer_comparison(
    src=source_audio,
    tgt=target_audio,
    out=output_audio,
    sr=sample_rate,
    N=N,
    H=H,
)

In [ ]:
# Inspect the SMS/HPS decomposition of source and target
import matplotlib.pyplot as plt
import numpy as np

src_dec = baseline_sms.decompose(source_audio)
tgt_dec = baseline_sms.decompose(target_audio)

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
fig.suptitle('SMS/HPS Decomposition', fontsize=14, fontweight='bold')

for row, (dec, label) in enumerate([(src_dec, 'Source'), (tgt_dec, 'Target')]):
    f0, sp, ap = dec

    # F0 contour
    axes[row, 0].plot(f0, color='steelblue', lw=1)
    axes[row, 0].set_title(f'{label} — F0')
    axes[row, 0].set_ylabel('F0 (Hz)')
    axes[row, 0].set_xlabel('Frame')

    # Spectral envelope
    axes[row, 1].imshow(sp.T, origin='lower', aspect='auto', cmap='magma')
    axes[row, 1].set_title(f'{label} — Spectral Envelope')
    axes[row, 1].set_ylabel('Frequency bin')
    axes[row, 1].set_xlabel('Frame')

    # Stochastic residual
    axes[row, 2].imshow(ap.T, origin='lower', aspect='auto', cmap='inferno')
    axes[row, 2].set_title(f'{label} — Stochastic Residual')
    axes[row, 2].set_ylabel('Coefficient')
    axes[row, 2].set_xlabel('Frame')

plt.tight_layout()
plt.show()

In [ ]:
from IPython.display import Audio, display, Markdown

display(Markdown("### Source Audio"))
display(Audio(source_audio, rate=sample_rate))

display(Markdown("### Target Audio"))
display(Audio(target_audio, rate=sample_rate))

display(Markdown(f"### Output Audio (method: `{method}`, alpha: `{alpha}`)"))
display(Audio(output_audio, rate=sample_rate))

In [ ]:
# Compare different alpha values
alphas = [0.25, 0.5, 0.75, 1.0]
transfer_fn = baseline_sms.f0_transfer if method == 'f0' else baseline_sms.f0_and_ap_transfer

results = {}
for a in alphas:
    results[a] = transfer_fn(target_audio, source_audio, alpha=a)
    print(f"alpha={a:.2f} done")

for a in alphas:
    display(Markdown(f"### alpha = {a}"))
    display(Audio(results[a], rate=sample_rate))